In [ ]:
import os
import sys

sys.path.append(os.path.abspath("../"))

import matplotlib.pyplot as plt
import tensorflow as tf

from util import dataset as u_dataset
from util import dataset_io as u_dataset_io
from util import image as u_image
from util import visualization as u_visualization

dataset_utils = u_dataset.DatasetUtils(u_dataset.DatasetConfig())

In [ ]:
directory = (
    "../../data/groundtruth/RC22_Sarah_Sarah_Default_RoboCup2022_FieldA__Nao-Devils_2ndHalf_5"
)

labels = u_dataset_io.load_labels(directory)

for sample in labels:
    if sample["intersections"]["ignore_sample"]:
        ignore_sample = sample
        break
    if not sample["intersections"]["ignore_sample"] and len(sample["intersections"]["L"]) > 0:
        intersections = sample

    if sample["name"] == "Upper7706":
        ball_sample = sample


print(intersections)
# print(ball_sample)
print(ignore_sample)

current_sample = intersections

In [ ]:
image = u_dataset_io.load_image(directory, current_sample, image_format=u_image.ImageFormat.RGB)

plt.imshow(image[...], cmap="gray")
plt.title("Raw Example Image in Grayscale")
plt.show()

In [ ]:
object_name = u_dataset.CategoryNames.INTERSECTIONS

test_coords = tf.constant([[189.48, 100.048], [183.48, 490.048], [-500, 30], [30, -500], [-500, -500]], tf.float32)

# u_visualization.show_masks_on_image(
#     directory=directory,
#     label=current_sample,
#     object_name=object_name.value,
#     mask_name="object",
# )

u_visualization.show_masks_on_image(
    coordinates=test_coords,
    image=image,
    mask_name="object",
)

In [ ]:
offset_mask = dataset_utils.get_masks(coordinates=test_coords)["offsets"]

coord_mask = dataset_utils.get_coordinate_mask(offset_mask)

# tf.print(tf.cast(coord_mask, tf.int16)[0:3, 0:3])

In [ ]:
train_ds = u_dataset_io.get_dataset(
    "../../data/groundtruth/GORE23_Gott_Gott_CompetitionWalk_Default__HTWK-Robots_1stHalf_1.tfrecords"
)

In [ ]:
for i in train_ds:
    print(i.keys())
    print("Ball keys: ", i["ball"].keys())
    print("PenaltyMark keys: ", i["penaltyMark"].keys())
    print("Intersection keys: ", i["intersections"].keys())

    print(tf.cast(i["intersections"]["object_mask"], tf.int8))
    print(tf.cast(i["intersections"]["loss_mask"], tf.int8))
    print(tf.cast(i["intersections"]["classification_mask"], tf.int8))
    break

In [ ]:
classification_mask = tf.expand_dims(
    dataset_utils.get_masks(current_sample, u_dataset.CategoryNames.INTERSECTIONS.value)[
        "classification_mask"
    ],
    axis=0,
)

print(tf.cast(classification_mask, tf.int16).shape)


one_hot_mask = dataset_utils.classification_mask_to_one_hot(
    classification_mask, u_dataset.CategoryNames.INTERSECTIONS.value
)
print(tf.reduce_sum(one_hot_mask))
print(one_hot_mask.shape[1] * one_hot_mask.shape[2])
print(tf.reduce_all(tf.reduce_sum(one_hot_mask, axis=-1) == tf.ones_like(tf.reduce_sum(one_hot_mask, axis=-1))))
tf.debugging.assert_equal(
    tf.reduce_sum(one_hot_mask),
    tf.cast(one_hot_mask.shape[1] * one_hot_mask.shape[2], dtype=one_hot_mask.dtype), message="Invalid one-hot classification mask."
)

print(tf.cast(one_hot_mask, tf.int16)[4:7, 10:13])